In [6]:
import pickle
import math
import re
import spacy
from collections import Counter, defaultdict

# 1. Inicializace spaCy (pokud už nemáte načtené)
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^\w\s]", " ", text.lower())
    return " ".join(token.lemma_ for token in nlp(text) if not token.is_space)

# 2. Načtení indexů z MetaCentra (z vašich .pkl souborů)
print("Načítám indexy...")
with open("index_title.pkl", "rb") as f:
    index_title = pickle.load(f)
with open("index_text.pkl", "rb") as f:
    index_text = pickle.load(f)

M = 240000 

def tf_idf_optimized(query, inverted_index, M):
    processed_query = preprocess(query)
    query_words = processed_query.split()
    query_tf = Counter(query_words)
    
    scores = defaultdict(float)
    
    for w, c_w_q in query_tf.items():
        if w in inverted_index:
            doc_counts = inverted_index[w]
            df_w = len(doc_counts)
            idf = math.log((M + 1) / df_w)
            
            for doc_id, c_w_d in doc_counts.items():
                scores[doc_id] += c_w_q * c_w_d * idf
                
    return scores

def search_wiki(query, alpha=0.7):
    scores_title = tf_idf_optimized(query, index_title, M)
    scores_text = tf_idf_optimized(query, index_text, M)
    
    combined_scores = {}
    all_doc_ids = set(scores_title.keys()).union(set(scores_text.keys()))
    
    for doc_id in all_doc_ids:
        s_title = scores_title.get(doc_id, 0.0)
        s_text = scores_text.get(doc_id, 0.0)
        combined_scores[doc_id] = alpha * s_title + (1 - alpha) * s_text
        
    sorted_results = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:5]

# --- 3. ZADÁNÍ VLASTNÍCH DOTAZŮ PRO UKÁZKU ---
vlastni_dotazy = [
    "albert einstein physics",
    "second world war history",
    "solar system planets",
    "black hole",
    "albert einstein",
    "physics"
]

for q in vlastni_dotazy:
    print(f"\n========================================")
    print(f"Výsledky pro dotaz: '{q}'")
    print(f"========================================")
    vysledky = search_wiki(q, alpha=0.7)
    for rank, (doc_id, score) in enumerate(vysledky, 1):
        print(f"{rank}. Dokument ID: {doc_id} | Skóre: {score:.4f}")

Načítám indexy...

Výsledky pro dotaz: 'albert einstein physics'
1. Dokument ID: 83534 | Skóre: 427.4769
2. Dokument ID: 524 | Skóre: 161.1732
3. Dokument ID: 308 | Skóre: 94.7288
4. Dokument ID: 2083 | Skóre: 57.4781
5. Dokument ID: 373 | Skóre: 44.8290

Výsledky pro dotaz: 'second world war history'
1. Dokument ID: 25634 | Skóre: 117.0195
2. Dokument ID: 56560 | Skóre: 111.3062
3. Dokument ID: 56559 | Skóre: 106.6228
4. Dokument ID: 2814 | Skóre: 103.2345
5. Dokument ID: 71396 | Skóre: 93.6839

Výsledky pro dotaz: 'solar system planets'
1. Dokument ID: 391 | Skóre: 166.5738
2. Dokument ID: 66407 | Skóre: 128.1584
3. Dokument ID: 177076 | Skóre: 109.5577
4. Dokument ID: 292 | Skóre: 96.2581
5. Dokument ID: 210 | Skóre: 94.2701

Výsledky pro dotaz: 'black hole'
1. Dokument ID: 818 | Skóre: 201.9090
2. Dokument ID: 1234 | Skóre: 88.8987
3. Dokument ID: 104910 | Skóre: 74.7906
4. Dokument ID: 48154 | Skóre: 64.0413
5. Dokument ID: 117326 | Skóre: 56.3007

Výsledky pro dotaz: 'albert eins